<a href="https://colab.research.google.com/github/aravindanmoorthy/Claude-Hackathon/blob/claude%2Fweather-alerts-parked-cars-58mry/safe_pilot_weather_alert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛡️ Safe Pilot — Weather Alert System
### USAA Hackathon Project

This notebook checks real-time weather for example GPS locations and determines whether a **Safe Pilot alert** should be triggered.

**API:** [Open-Meteo](https://open-meteo.com/) — Free, no API key required!

---
**Scenarios covered:**
- 🅿️ **Scenario 1** — Vehicle parked: alert driver before severe weather arrives
- 🚗 **Scenario 2** — Vehicle moving: monitor weather 30 miles ahead continuously

> Run each cell top to bottom using **Shift+Enter**

In [1]:
# requests is pre-installed in Colab, but just in case
!pip install requests --quiet
print('✅ Dependencies ready!')

✅ Dependencies ready!


In [2]:
import requests
from dataclasses import dataclass
from datetime import datetime

print('✅ Imports successful!')

✅ Imports successful!


In [3]:
# Weather codes that trigger a Safe Pilot alert
ALERT_THRESHOLD_CODES = {95, 96, 99, 82, 75, 65, 45, 48}

SEVERE_WEATHER_CODES = {
    45: 'Foggy',
    48: 'Icy Fog',
    61: 'Light Rain',
    63: 'Moderate Rain',
    65: 'Heavy Rain',
    71: 'Light Snow',
    73: 'Moderate Snow',
    75: 'Heavy Snow',
    80: 'Rain Showers',
    81: 'Heavy Showers',
    82: 'Violent Showers',
    95: 'Thunderstorm',
    96: 'Thunderstorm + Hail',
    99: 'Thunderstorm + Heavy Hail',
}

ALERT_SEVERITY = {
    45: 'MEDIUM',
    48: 'MEDIUM',
    65: 'MEDIUM',
    75: 'HIGH',
    82: 'HIGH',
    95: 'HIGH',
    96: 'HIGH',
    99: 'CRITICAL',
}

print('✅ Constants loaded!')

✅ Constants loaded!


In [4]:
EXAMPLE_LOCATIONS = [
    {'name': 'Edinburg, TX (South Texas)',       'lat': 26.3017,  'lon': -98.1633},
    {'name': 'Miami, FL (Storm-prone)',           'lat': 25.7617,  'lon': -80.1918},
    {'name': 'Denver, CO (Mountain Weather)',     'lat': 39.7392,  'lon': -104.9903},
    {'name': 'Oklahoma City, OK (Tornado Alley)', 'lat': 35.4676,  'lon': -97.5164},
    {'name': 'San Diego, CA (Usually Clear)',     'lat': 32.7157,  'lon': -117.1611},
]

print(f'✅ {len(EXAMPLE_LOCATIONS)} locations ready!')
for loc in EXAMPLE_LOCATIONS:
    print(f"   📍 {loc['name']} → lat: {loc['lat']}, lon: {loc['lon']}")

✅ 5 locations ready!
   📍 Edinburg, TX (South Texas) → lat: 26.3017, lon: -98.1633
   📍 Miami, FL (Storm-prone) → lat: 25.7617, lon: -80.1918
   📍 Denver, CO (Mountain Weather) → lat: 39.7392, lon: -104.9903
   📍 Oklahoma City, OK (Tornado Alley) → lat: 35.4676, lon: -97.5164
   📍 San Diego, CA (Usually Clear) → lat: 32.7157, lon: -117.1611


In [5]:
@dataclass
class WeatherAlert:
    location_name:    str
    lat:              float
    lon:              float
    is_severe_now:    bool
    condition_now:    str
    weather_code:     int
    wind_speed_mph:   float
    visibility_miles: float
    temperature_f:    float
    upcoming_alerts:  list
    alert_required:   bool
    severity:         str
    alert_message:    str

print('✅ WeatherAlert data class defined!')

✅ WeatherAlert data class defined!


In [6]:
def fetch_weather(lat: float, lon: float) -> dict:
    """Call Open-Meteo API with lat/lon and return raw JSON response."""
    url = 'https://api.open-meteo.com/v1/forecast'
    params = {
        'latitude':  lat,
        'longitude': lon,
        'current': [
            'temperature_2m', 'wind_speed_10m',
            'weather_code', 'precipitation', 'visibility'
        ],
        'hourly': [
            'precipitation_probability', 'wind_gusts_10m',
            'weather_code', 'visibility'
        ],
        'temperature_unit': 'fahrenheit',
        'wind_speed_unit':  'mph',
        'forecast_days': 1
    }
    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f'  ⚠️  API call failed: {e}')
        return None

print('✅ fetch_weather() defined!')

✅ fetch_weather() defined!


In [7]:
def parse_weather_alert(location: dict, response: dict) -> WeatherAlert:
    """Parse raw API response into a structured WeatherAlert object."""
    current = response['current']
    hourly  = response['hourly']

    weather_code     = current['weather_code']
    wind_speed_mph   = current['wind_speed_10m']
    visibility_mi    = current['visibility'] / 1609
    temperature_f    = current['temperature_2m']
    condition_now    = SEVERE_WEATHER_CODES.get(weather_code, 'Clear / Partly Cloudy')
    is_severe_now    = weather_code in ALERT_THRESHOLD_CODES

    # Look 1 hour ahead
    upcoming_alerts = []
    for i, code in enumerate(hourly['weather_code'][:1]):
        if code in ALERT_THRESHOLD_CODES:
            upcoming_alerts.append({
                'hour':       hourly['time'][i],
                'condition':  SEVERE_WEATHER_CODES[code],
                'rain_prob':  hourly['precipitation_probability'][i],
                'wind_gusts': hourly['wind_gusts_10m'][i],
            })

    alert_required = is_severe_now or len(upcoming_alerts) > 0

    severity = 'NONE'
    if is_severe_now:
        severity = ALERT_SEVERITY.get(weather_code, 'MEDIUM')
    elif upcoming_alerts:
        severity = ALERT_SEVERITY.get(hourly['weather_code'][0], 'MEDIUM')

    if alert_required:
        if is_severe_now:
            alert_message = (
                f'🚨 SAFE PILOT ALERT [{severity}]: {condition_now} detected at your location! '
                f'Wind: {wind_speed_mph:.1f} mph | Visibility: {visibility_mi:.1f} miles. '
                f'Please move your vehicle to a safe location immediately.'
            )
        else:
            u = upcoming_alerts[0]
            alert_message = (
                f"⚠️  SAFE PILOT ALERT [{severity}]: {u['condition']} expected within 1 hour "
                f"(Rain probability: {u['rain_prob']}% | Wind gusts: {u['wind_gusts']:.1f} mph). "
                f'Consider moving your vehicle to a safe location soon.'
            )
    else:
        alert_message = '✅ No alert needed. Weather conditions are safe.'

    return WeatherAlert(
        location_name    = location['name'],
        lat              = location['lat'],
        lon              = location['lon'],
        is_severe_now    = is_severe_now,
        condition_now    = condition_now,
        weather_code     = weather_code,
        wind_speed_mph   = wind_speed_mph,
        visibility_miles = visibility_mi,
        temperature_f    = temperature_f,
        upcoming_alerts  = upcoming_alerts,
        alert_required   = alert_required,
        severity         = severity,
        alert_message    = alert_message,
    )

print('✅ parse_weather_alert() defined!')

✅ parse_weather_alert() defined!


In [8]:
def print_alert(alert: WeatherAlert):
    print('=' * 65)
    print(f'  📍 {alert.location_name}')
    print(f'     Lat: {alert.lat} | Lon: {alert.lon}')
    print('-' * 65)
    print(f'  🌡️  Temperature   : {alert.temperature_f:.1f} F')
    print(f'  🌤️  Condition Now : {alert.condition_now} (code {alert.weather_code})')
    print(f'  💨  Wind Speed    : {alert.wind_speed_mph:.1f} mph')
    print(f'  👁️  Visibility    : {alert.visibility_miles:.1f} miles')
    if alert.upcoming_alerts:
        print('\n  📅 Upcoming (next 1 hour):')
        for a in alert.upcoming_alerts:
            print(f"     • {a['hour']} → {a['condition']} | Rain: {a['rain_prob']}% | Gusts: {a['wind_gusts']:.1f} mph")
    print(f'\n  {alert.alert_message}')
    print('=' * 65)
    print()

print('✅ print_alert() defined!')

✅ print_alert() defined!


In [9]:
def check_weather_at_location(location: dict):
    print(f"\n🔍 Checking weather for: {location['name']} ...")
    response = fetch_weather(location['lat'], location['lon'])
    if response is None:
        print('  ❌ Skipping — could not retrieve weather data.')
        return None
    alert = parse_weather_alert(location, response)
    print_alert(alert)
    return alert

print('✅ check_weather_at_location() defined!')

✅ check_weather_at_location() defined!


In [10]:
print('\n' + '🛡️  SAFE PILOT — Weather Alert System'.center(65))
print(f"{'Run at: ' + datetime.now().strftime('%Y-%m-%d %H:%M:%S'):^65}\n")

alerts_triggered = []
no_alerts        = []

for location in EXAMPLE_LOCATIONS:
    alert = check_weather_at_location(location)
    if alert:
        if alert.alert_required:
            alerts_triggered.append(alert.location_name)
        else:
            no_alerts.append(alert.location_name)

print('\n' + '-' * 65)
print('📊 SUMMARY')
print('-' * 65)
if alerts_triggered:
    print(f'  🚨 Alerts Triggered ({len(alerts_triggered)}):')
    for name in alerts_triggered:
        print(f'     • {name}')
if no_alerts:
    print(f'  ✅ No Alert Needed ({len(no_alerts)}):')
    for name in no_alerts:
        print(f'     • {name}')
print('-' * 65)


              🛡️  SAFE PILOT — Weather Alert System              
                   Run at: 2026-03-31 19:44:04                   


🔍 Checking weather for: Edinburg, TX (South Texas) ...
  📍 Edinburg, TX (South Texas)
     Lat: 26.3017 | Lon: -98.1633
-----------------------------------------------------------------
  🌡️  Temperature   : 88.4 F
  🌤️  Condition Now : Clear / Partly Cloudy (code 0)
  💨  Wind Speed    : 14.5 mph
  👁️  Visibility    : 24.4 miles

  ✅ No alert needed. Weather conditions are safe.


🔍 Checking weather for: Miami, FL (Storm-prone) ...
  📍 Miami, FL (Storm-prone)
     Lat: 25.7617 | Lon: -80.1918
-----------------------------------------------------------------
  🌡️  Temperature   : 79.1 F
  🌤️  Condition Now : Clear / Partly Cloudy (code 1)
  💨  Wind Speed    : 13.5 mph
  👁️  Visibility    : 15.8 miles

  ✅ No alert needed. Weather conditions are safe.


🔍 Checking weather for: Denver, CO (Mountain Weather) ...
  📍 Denver, CO (Mountain Weather)
     Lat: 3

In [11]:
# Change these to any location you want to test!
custom_location = {
    'name': 'My Custom Location',
    'lat':  26.3017,   # Replace with your latitude
    'lon': -98.1633    # Replace with your longitude
}

check_weather_at_location(custom_location)


🔍 Checking weather for: My Custom Location ...
  📍 My Custom Location
     Lat: 26.3017 | Lon: -98.1633
-----------------------------------------------------------------
  🌡️  Temperature   : 88.4 F
  🌤️  Condition Now : Clear / Partly Cloudy (code 0)
  💨  Wind Speed    : 14.5 mph
  👁️  Visibility    : 24.4 miles

  ✅ No alert needed. Weather conditions are safe.



WeatherAlert(location_name='My Custom Location', lat=26.3017, lon=-98.1633, is_severe_now=False, condition_now='Clear / Partly Cloudy', weather_code=0, wind_speed_mph=14.5, visibility_miles=24.362958359229335, temperature_f=88.4, upcoming_alerts=[], alert_required=False, severity='NONE', alert_message='✅ No alert needed. Weather conditions are safe.')